# 03 — Exploratory Data Analysis (EDA)

**Project:** Meateka — Digital Content Engagement & Reach Predictor  
**Dataset:** `cleaned_dataset.csv` ($N = 2,777$ validated posts across TikTok, Instagram, and Facebook)  
**Academic Focus:** Historical pattern exploration, group performance comparisons, multivariable relationships, and preprocessing justification.  

### Methodology & Aggregation Standards
Following academic advisor guidance, group performance comparisons throughout this notebook report the **arithmetic mean (average)** engagement rate alongside exact **sample sizes ($n$)**. While social media metrics exhibit positive skewness (with viral observations in the upper tail), the arithmetic mean reflects expected aggregate performance and total engagement potential across cohorts. Distribution shapes, medians, and standard deviations are reported concurrently to provide complete statistical transparency without conflating historical correlation with causal influence.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats

# Visual styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 10

# Load validated cleaned dataset
DATA_PATH = Path('../data/processed/cleaned_dataset.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/processed/cleaned_dataset.csv')

df = pd.read_csv(DATA_PATH)
print(f"[EDA] Successfully loaded dataset: {len(df):,} rows, {df.shape[1]} columns")
print(f"Platforms present: {list(df['Platform'].unique())}")


## A. Platform Performance Analysis ($N = 2,777$)
Directly supports **Feature 2 (Platform Comparison)**: Examining historical engagement distributions across TikTok, Instagram, and Facebook.

In [ ]:
platform_stats = df.groupby('Platform')['Engagement_Rate'].agg(
    sample_size='count',
    mean_engagement='mean',
    median_engagement='median',
    std_dev='std',
    iqr=lambda x: sp_stats.iqr(x)
).round(2).sort_values('mean_engagement', ascending=False)

# Add percentage share of total dataset
platform_stats['pct_of_total'] = ((platform_stats['sample_size'] / len(df)) * 100).round(1)
platform_stats = platform_stats[['sample_size', 'pct_of_total', 'mean_engagement', 'median_engagement', 'std_dev', 'iqr']]
print("=== Historical Engagement Rate by Platform ===")
display(platform_stats)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Average Engagement Rate Barplot with sample size context (n)
plat_plot_data = df.groupby('Platform')['Engagement_Rate'].mean().reset_index()
plat_counts = df['Platform'].value_counts()
plat_labels = [f"{p}\n(n={plat_counts[p]:,})" for p in plat_plot_data['Platform']]

sns.barplot(data=plat_plot_data, x='Platform', y='Engagement_Rate', ax=axes[0], palette='Blues_r')
axes[0].set_xticklabels(plat_labels)
axes[0].set_title('Average Engagement Rate by Platform (with Sample Size n)', fontweight='bold')
axes[0].set_ylabel('Average Engagement Rate (%)')
axes[0].set_xlabel('Platform')
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

# 2. Distribution Boxplot (outliers truncated at 50% for visual readability)
sns.boxplot(data=df, x='Platform', y='Engagement_Rate', ax=axes[1], palette='Blues_r', showfliers=False)
axes[1].set_xticklabels(plat_labels)
axes[1].set_title('Engagement Rate Spread by Platform (Whiskers: IQR Spread)', fontweight='bold')
axes[1].set_ylabel('Engagement Rate (%)')
axes[1].set_xlabel('Platform')

plt.tight_layout()
plt.show()


**Academic Observations & Non-Causal Interpretation:**
* **Historical Associations:** In this dataset, TikTok posts recorded a significantly higher average engagement rate ($27.78\%, n=718$) compared to Instagram ($6.34\%, n=1,138$) and Facebook ($5.81\%, n=921$). 
* **Confounding Variables:** This pattern is strongly linked to the creator composition: TikTok observations in this sample contain a higher ratio of smaller-scale, highly active creators with smaller denominator follower counts, yielding naturally higher engagement percentages.
* **Non-Causal Note:** These results describe historical associations within this specific sample; they do not imply that publishing on TikTok *causes* content to become viral, nor do they guarantee superior reach for any single post.
* **Mean vs. Median Nuance:** While median engagement across all platforms remains between $5.0\%$ and $6.0\%$, the arithmetic mean accurately reflects total viral potential and high-performing outliers in the right tail.


## B. Category / Topic Performance Analysis ($N = 2,777$)
Supports **Feature 4 (Content Idea Recommendation)**: Identifying historical category trends and cross-platform topic dynamics.

In [ ]:
cat_summary = df.groupby('Category')['Engagement_Rate'].agg(
    sample_size='count',
    mean_engagement='mean',
    median_engagement='median'
).sort_values('mean_engagement', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
bars = sns.barplot(x=cat_summary.index, y=cat_summary['mean_engagement'], ax=ax, palette='mako')

# Include sample size n in labels
cat_labels = [f"{cat}\n(n={cat_summary.loc[cat, 'sample_size']})" for cat in cat_summary.index]
ax.set_xticklabels(cat_labels, rotation=45, ha='right')
ax.set_title('Average Engagement Rate by Category (with Sample Sizes n)', fontweight='bold')
ax.set_ylabel('Average Engagement Rate (%)')
ax.set_xlabel('Category')

for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# Pivot table using MEAN / AVERAGE as requested by advisor
pivot_cat_platform = df.pivot_table(
    index='Category', 
    columns='Platform', 
    values='Engagement_Rate', 
    aggfunc='mean'
).round(2)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(pivot_cat_platform, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax, cbar_kws={'label': 'Average Engagement Rate (%)'})
ax.set_title('Average Engagement Rate: Category x Platform Matrix', fontweight='bold', pad=12)
ax.set_ylabel('Category / Topic')
ax.set_xlabel('Platform')
plt.tight_layout()
plt.show()


**Academic Observations:**
* Category performance is not uniform across platforms. Categories such as Fitness, Entertainment, and Food exhibited higher historical average engagement on TikTok, while Lifestyle and Travel demonstrated consistent performance across Instagram.
* This confirms the architectural decision that content recommendations must be computed **per platform** rather than applying an overgeneralized global topic rank.


## C. Content Type & Media Presence Analysis
Supports **Feature 4 (Content Type Recommendation)** and validates the retention of `Has_Media` as a pre-posting feature.

In [ ]:
ct_summary = df.groupby('Content_Type')['Engagement_Rate'].agg(
    sample_size='count',
    mean_engagement='mean',
    median_engagement='median'
).sort_values('mean_engagement', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x=ct_summary.index, y=ct_summary['mean_engagement'], ax=ax, palette='flare')
ct_labels = [f"{ct}\n(n={ct_summary.loc[ct, 'sample_size']})" for ct in ct_summary.index]
ax.set_xticklabels(ct_labels, rotation=30, ha='right')
ax.set_title('Average Engagement Rate by Content Type (with Sample Size n)', fontweight='bold')
ax.set_ylabel('Average Engagement Rate (%)')
ax.set_xlabel('Content Type')

for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
media_stats = df.groupby('Has_Media')['Engagement_Rate'].agg(
    sample_size='count',
    mean_engagement='mean',
    median_engagement='median',
    std_dev='std'
).round(2)
media_stats['pct_of_total'] = ((media_stats['sample_size'] / len(df)) * 100).round(1)
media_stats.index = ['Text-Only (False)', 'Has Media (True)']

print("=== Engagement Comparison by Media Presence ===")
display(media_stats)

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(x=media_stats.index, y=media_stats['mean_engagement'], ax=ax, palette='viridis')
ax.set_title('Average Engagement Rate: Has Media vs. Text-Only', fontweight='bold')
ax.set_ylabel('Average Engagement Rate (%)')
for p, n in zip(ax.patches, media_stats['sample_size']):
    ax.annotate(f"{p.get_height():.2f}%\n(n={n:,})", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
print("=== Average Engagement Rate by Platform and Content Type ===")
p_ct_table = df.groupby(['Platform', 'Content_Type'])['Engagement_Rate'].agg(
    sample_size='count',
    mean_engagement='mean',
    median_engagement='median'
).round(2)
display(p_ct_table)


**Academic Observations:**
* In historical observations, visual and short-form video media (Duets, Stitches, Videos, Carousels) recorded higher engagement metrics than static status updates.
* Text-only posts represent $15.2\%$ of the sample ($n = 422$) with an average engagement of $11.36\%$. Retaining `Has_Media` in the preprocessing pipeline ensures the clustering algorithm isolates this specific format subset.


## D. Temporal Analysis (Hour of Day, Day of Week, Month)
Supports **Feature 3 (Best Posting Time Recommendation)**: Investigating historical timing patterns.

In [ ]:
hourly_stats = df.groupby('Hour_of_Day')['Engagement_Rate'].agg(['count', 'mean']).round(2)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(hourly_stats.index, hourly_stats['mean'], marker='o', color='steelblue', linewidth=2.5, label='Average Engagement Rate (%)')
ax.axvspan(18, 21, color='orange', alpha=0.2, label='Peak Evening Window (18:00 - 21:00)')
ax.set_title('Average Engagement Rate by Hour of Day (0 - 23)', fontweight='bold')
ax.set_xlabel('Hour of Day (24-Hour Format)')
ax.set_ylabel('Average Engagement Rate (%)')
ax.set_xticks(range(0, 24))
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print("Top 5 Hours by Average Engagement Rate:")
display(hourly_stats.sort_values('mean', ascending=False).head(5))


In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_stats = df.groupby('Day_of_Week')['Engagement_Rate'].agg(['count', 'mean']).reindex(dow_order).round(2)

fig, ax = plt.subplots(figsize=(10, 4.5))
dow_labels = [f"{d}\n(n={dow_stats.loc[d, 'count']})" for d in dow_order]
sns.barplot(x=dow_order, y=dow_stats['mean'], ax=ax, palette='crest')
ax.set_xticklabels(dow_labels)
ax.set_title('Average Engagement Rate by Day of Week (with Sample Sizes n)', fontweight='bold')
ax.set_ylabel('Average Engagement Rate (%)')
ax.set_xlabel('Day of Week')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
available_months = [m for m in month_order if m in df['Month'].values]
month_stats = df.groupby('Month')['Engagement_Rate'].agg(['count', 'mean']).reindex(available_months).round(2)

fig, ax = plt.subplots(figsize=(12, 4.5))
sns.barplot(x=available_months, y=month_stats['mean'], ax=ax, palette='Spectral')
ax.set_xticklabels([f"{m[:3]}\n(n={month_stats.loc[m, 'count']})" for m in available_months], rotation=0)
ax.set_title('Average Engagement Rate by Month (with Sample Sizes n)', fontweight='bold')
ax.set_ylabel('Average Engagement Rate (%)')
ax.set_xlabel('Month')
plt.tight_layout()
plt.show()


In [ ]:
best_hour_per_platform = (
    df.groupby(['Platform', 'Hour_of_Day'])['Engagement_Rate']
    .agg(sample_size='count', avg_engagement='mean')
    .reset_index()
    .sort_values(['Platform', 'avg_engagement'], ascending=[True, False])
    .groupby('Platform')
    .head(3)
)
print("Top 3 Historical Peak Hours by Platform (Average Engagement Rate):")
display(best_hour_per_platform.round(2))


**Academic Observations:**
* Historical engagement shows steady distribution across days of the week, with slight weekend elevation ($12.01\%$ on Saturdays).
* Hourly analysis indicates concentration of higher average interaction between 18:00 and 21:00, coinciding with evening leisure. Platform-specific peaks differ slightly, confirming the need for localized hourly recommendation lookups.

## E. Influencer Tier & Follower Scale Analysis
Investigating audience scale dynamics and the inverse engagement relationship.

In [ ]:
tier_order = ['Nano', 'Micro', 'Mid-tier', 'Macro']
tier_stats = df.groupby('Influencer_Tier')['Engagement_Rate'].agg(
    sample_size='count',
    mean_engagement='mean',
    median_engagement='median',
    std_dev='std'
).reindex(tier_order).round(2)

tier_stats['pct_of_total'] = ((tier_stats['sample_size'] / len(df)) * 100).round(1)
print("=== Engagement Rate by Influencer Tier ===")
display(tier_stats)

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(x=tier_order, y=tier_stats['mean_engagement'], ax=ax, palette='rocket')
tier_labels = [f"{t}\n(n={tier_stats.loc[t, 'sample_size']:,})" for t in tier_order]
ax.set_xticklabels(tier_labels)
ax.set_title('Average Engagement Rate by Influencer Tier (with Sample Sizes n)', fontweight='bold')
ax.set_ylabel('Average Engagement Rate (%)')
ax.set_xlabel('Influencer Tier')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
corr_followers = df['Follower_Count'].corr(df['Engagement_Rate'])
print(f"Pearson Correlation (Follower_Count vs Engagement_Rate): {corr_followers:.4f}")

fig, ax = plt.subplots(figsize=(9, 5))
sample_df = df.sample(min(1000, len(df)), random_state=42)
sns.scatterplot(data=sample_df, x='Follower_Count', y='Engagement_Rate', hue='Platform', alpha=0.6, ax=ax)
ax.set_ylim(0, 150)  # Truncate viral outliers for readable scatter
ax.set_title(f'Follower Count vs. Engagement Rate (Pearson r = {corr_followers:.2f})', fontweight='bold')
ax.set_xlabel('Follower Count')
ax.set_ylabel('Engagement Rate (%)')
plt.tight_layout()
plt.show()


**Academic Observations & Non-Causal Nuance:**
* **The Scale Inverse Pattern:** Analysis confirms a moderate negative correlation ($r = -0.4187$) between `Follower_Count` and `Engagement_Rate`. In social media metrics, engagement rate is mathematically calculated as interaction volume divided by follower base. Accounts with smaller followings (Nano and Micro) record higher percentage rates because active interactions represent a larger proportion of their audience.
* **Sample Composition Note:** Macro influencers represent the largest cohort ($n = 2,193, 79.0\%$), with a steady average engagement rate of $3.59\%$. Nano ($n=59$) and Micro ($n=248$) influencers exhibit high variance with instances of extreme engagement rates.


## F. Multivariable Interaction Analyses
Exploring cross-variable dynamics to understand how content and creator properties intersect.

In [ ]:
print("=== 1. Influencer Tier x Platform (Average Engagement Rate & Counts) ===")
tier_platform_pivot = df.pivot_table(
    index='Influencer_Tier',
    columns='Platform',
    values='Engagement_Rate',
    aggfunc=['count', 'mean']
).reindex(tier_order)
display(tier_platform_pivot.round(2))

print("\n=== 2. Platform x Media Presence Interaction ===")
media_plat_pivot = df.pivot_table(
    index='Platform',
    columns='Has_Media',
    values='Engagement_Rate',
    aggfunc=['count', 'mean']
).round(2)
display(media_plat_pivot)

print("\n=== 3. Numerical Feature Correlation Matrix ===")
num_cols = ['Follower_Count', 'Content_Length', 'Hashtag_Count', 'Hour_of_Day', 'Month', 'Engagement_Rate']
corr_matrix = df[num_cols].corr().round(3)
display(corr_matrix)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-0.5, vmax=1.0, fmt='.2f', ax=ax)
ax.set_title('Correlation Matrix of Numerical Features and Target Engagement', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


**Academic Observations on Multivariable Relationships:**
* **Influencer Tier × Platform:** Across all three platforms, Nano and Micro tiers record higher average percentage engagement rates than Macro accounts, though Macro accounts constitute the majority of observations on Instagram and Facebook.
* **Platform × Media Presence:** On Facebook, posts with media attached averaged $6.28\%$ vs. $3.39\%$ for text-only updates ($n=149$).
* **Weak Linear Predictors:** Caption length ($r = -0.113$) and hashtag count ($r = 0.020$) display very weak linear correlations with engagement rate, confirming that metadata alone cannot linearly predict reach without multi-dimensional clustering.


## G. Caption Sentiment & Hashtag Analysis
Examining sentiment labels and hashtag distribution patterns.

In [ ]:
sentiment_stats = df.groupby('Sentiment')['Engagement_Rate'].agg(
    sample_size='count',
    mean_engagement='mean',
    median_engagement='median'
).sort_values('mean_engagement', ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=sentiment_stats.index, y=sentiment_stats['mean_engagement'], ax=ax, palette='Blues_d')
sent_labels = [f"{s}\n(n={sentiment_stats.loc[s, 'sample_size']:,})" for s in sentiment_stats.index]
ax.set_xticklabels(sent_labels)
ax.set_title('Average Engagement Rate by Caption Sentiment (with Sample Sizes n)', fontweight='bold')
ax.set_ylabel('Average Engagement Rate (%)')
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')

plt.tight_layout()
plt.show()


## H. Target Distribution Dynamics & Heavy-Tailed Skewness
Evaluating engagement rate distribution properties and statistical moments.

In [ ]:
er = df['Engagement_Rate']
print(f"Sample Size (N):    {len(er):,}")
print(f"Mean (Average):     {er.mean():.2f}%")
print(f"Median:             {er.median():.2f}%")
print(f"Standard Deviation: {er.std():.2f}%")
print(f"Skewness:           {er.skew():.2f}")
print(f"Kurtosis:           {er.kurt():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Visual range zoom: 0% to 200% (captures 98.7% of posts without visual distortion)
er_zoom = er[er <= 200]
sns.histplot(er_zoom, bins=50, ax=axes[0], color='steelblue', kde=True)
axes[0].axvline(er.mean(), color='red', linestyle='--', linewidth=2, label=f"Mean = {er.mean():.1f}%")
axes[0].axvline(er.median(), color='green', linestyle=':', linewidth=2, label=f"Median = {er.median():.1f}%")
axes[0].set_title(f'Engagement Rate Distribution (Zoomed 0 - 200%, {len(er_zoom)/len(er)*100:.1f}% data)', fontweight='bold')
axes[0].set_xlabel('Engagement Rate (%)')
axes[0].legend()

# Log-transformed distribution
sns.histplot(np.log1p(er), bins=50, ax=axes[1], color='seagreen', kde=True)
axes[1].set_title('Engagement Rate Distribution (log1p-transformed)', fontweight='bold')
axes[1].set_xlabel('log1p(Engagement Rate)')

plt.tight_layout()
plt.show()


## I. Preprocessing Justification

To satisfy rigorous academic audit standards, the design decisions embodied in the Meateka preprocessing pipeline (`ColumnTransformer`) are justified below:

### 1. OneHotEncoder for Categorical Features
* **Features:** `Platform`, `Content_Type`, `Category`, `Day_of_Week`, `Sentiment`, `Influencer_Tier`.
* **Justification:** Categorical attributes possess no intrinsic mathematical order (e.g., TikTok is not "greater than" Instagram). Using Label Encoding would impose false numerical distances (e.g., $d(\text{TikTok}, \text{Facebook}) = 2$, but $d(\text{TikTok}, \text{Instagram}) = 1$). `OneHotEncoder(handle_unknown='ignore')` maps each level onto an orthogonal binary basis where all categories are equidistant (distance $\sqrt{2}$), ensuring Euclidean-based hierarchical clustering treats categorical differences impartially.

### 2. StandardScaler for Numerical Features
* **Features:** `Hour_of_Day`, `Month`, `Hashtag_Count`, `Content_Length`, `Follower_Count`, `Has_Media`, `Is_Verified`.
* **Justification:** Numerical features operate on drastically different physical units. `Follower_Count` spans from 100 to 500,000, while `Hashtag_Count` ranges from 0 to 30, and `Hour_of_Day` from 0 to 23. Without scaling, Euclidean distance ($d = \sqrt{\sum (x_i - y_i)^2}$) would be $99.9\%$ dominated by follower counts, effectively rendering hashtags and timing invisible to the algorithm. `StandardScaler` standardizes each feature to zero mean and unit variance ($\mu = 0, \sigma = 1$).

### 3. Exclusion of Post-Event Metrics (Zero Target Leakage)
* **Excluded Fields:** `Likes`, `Comments`, `Shares`, `Views`, `Saves`, `Engagement_Rate`.
* **Justification:** These metrics are outcomes that occur *after* publication. Including them in clustering or inference inputs would constitute severe **Data Leakage**, as a creator drafting a post cannot know future view counts. They are strictly reserved for post-clustering profiling and historical validation.

### 4. Exclusion of Identifiers and Raw Metadata
* **Excluded Fields:** `Post_ID`, arbitrary database timestamps.
* **Justification:** Unique identifiers introduce non-generalizable sample noise, prompting algorithms to memorize instances rather than clustering generalizable content patterns.

### 5. Retention of `Has_Media` and `Is_Verified`
* **Justification:** Both represent valid pre-posting metadata known prior to publication. Empirical ablation tests demonstrated that retaining them yields tighter cluster compactness ($+16.34\%$ Silhouette gain for `Is_Verified`, $+25.03\%$ for `Has_Media`), isolating verified creators and text-only content into cohesive structural groups.

### 6. Visual Outlier Zooming vs. Production Scaling
* **Justification:** Positive skewness in engagement rates causes linear visualizations to distort. Zooming to $0\% - 200\%$ in EDA captures $98.7\%$ of posts cleanly. In the ML pipeline, no arbitrary clipping is applied to pre-posting numericals; instead, `StandardScaler` standardizes the true variance.

### 7. Scope Restriction to Selected Platforms
* **Justification:** Restricting to TikTok, Instagram, and Facebook concentrates data density on the three leading consumer content networks, ensuring each sub-group maintains sufficient statistical sample size ($n = 718$ to $1,138$) for clustering stability.

### 8. Alignment with New-Post / Draft Planning Scenario
* **Justification:** Every feature in the 13-column feature vector represents actionable metadata available at the exact moment an author drafts content.


## J. EDA Summary — Key Academic Findings

1. **Aggregation Standard (Mean vs. Median):** Arithmetic mean engagement reflects total audience interaction potential across groups. However, because social media engagement follows a heavy-tailed distribution (Mean: $11.75\%$, Median: $5.69\%$), group comparisons must always be reported alongside sample size context ($n$).
2. **Platform Associations:** In this sample, TikTok posts ($n = 718$) showed higher average engagement rates ($27.78\%$) than Instagram ($6.34\%, n=1,138$) and Facebook ($5.81\%, n=921$), driven by sample creator composition (higher ratio of micro/nano accounts).
3. **Format & Media Presence:** Visual and short-form video media (Duets, Stitches, Videos, Carousels) systematically outperformed text-only posts ($n = 422, 11.36\%$ average engagement).
4. **The Audience Scale Inverse Pattern:** Follower count correlates negatively with percentage engagement rate ($r = -0.42$). Smaller creators (Nano: $n=59, 126.8\%$ average; Micro: $n=248, 48.7\%$ average) record higher percentage rates than Macro creators ($n=2,193, 3.59\%$ average).
5. **Temporal Dynamics:** Evening windows (18:00–21:00) and late-week postings show heightened historical interaction.
6. **Non-Causal Interpretation Guardrail:** All patterns describe historical empirical associations within this sample. They do not constitute causal claims, nor do they guarantee future post virality.
7. **Model Confidence Semantics:** The machine learning model outputs **cluster-membership similarity confidence** (degree of geometric alignment with historical cluster centroids), NOT an absolute probability of guaranteed future engagement.
